# 实时遥测、关节与服务状态

实时读取机器人状态：主程序内部 1Hz 轮询的关节舵机位置（T:106）、电池、主机传感器、主机错误环，以及语音 / VLM 服务状态。

网页 **Telemetry** 标签页展示同样的数据。

In [ ]:
import requests, time

BASE = "http://127.0.0.1:5000"
LABELS = ["LF_A", "LF_B", "LF_H", "LH_A", "LH_B", "LH_H",
          "RF_A", "RF_B", "RF_H", "RH_A", "RH_B", "RH_H"]

for i in range(5):
    d = requests.get(BASE + "/api/telemetry", timeout=3).json()
    fb = (d.get("joints") or {}).get("fb") or []
    print(f"--- 第 {i+1} 帧  电池={d.get('battery')}V  CPU={d.rpi.cpu_load}% ---")
    for name, v in zip(LABELS, fb):
        bar = "#" * (v * 30 // 1023)
        print(f"{name:5s} {v:4d} {bar}")
    time.sleep(1.0)

In [ ]:
d = requests.get(BASE + "/api/telemetry", timeout=3).json()
errs = d.get("errors", [])
print("主机错误:", len(errs))
for e in errs[:10]:
    print(f"[{e['level']}] {e['source']}: {e['msg'][:120]}")

## 板载服务

主板上运行三个服务：主程序（cron @reboot）、语音助手与 VLM 目标服务（systemd）。

In [ ]:
import subprocess
for svc in ("wavego-voice", "wavego-vlm"):
    r = subprocess.run(["systemctl", "is-active", svc], capture_output=True, text=True)
    print(f"{svc}: {r.stdout.strip()}")
print("主程序: cron @reboot -> 查看 ~/ugv.log")